In [12]:
#importing the libraries
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.callbacks import ModelCheckpoint


In [13]:
train_data = pd.read_csv("phm_train.csv")
test_data  = pd.read_csv("phm_test.csv")

In [14]:
#Declaring the english stop words
nltk.download('stopwords')
english_stops = set(stopwords.words('english'))


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\User\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [15]:
#data preprocessing
def preprocess(text_series):
    text_series = text_series.str.lower()
    text_series = text_series.replace({'<.*?>': ''}, regex=True)
    text_series = text_series.replace({'[^a-zA-Z]': ' '}, regex=True)
    text_series = text_series.apply(
        lambda x: ' '.join([word for word in x.split() if word not in english_stops])
    )
    return text_series

x_train = preprocess(train_data['tweet'])
x_test  = preprocess(test_data['tweet'])

y_train = train_data['label']
y_test  = test_data['label']


In [16]:
#sequence max length functioon
def get_max_length():
    lengths = [len(text.split()) for text in x_train]
    return int(np.ceil(np.mean(lengths)))

max_length = get_max_length()

In [17]:
#encode review
token = Tokenizer(lower=False)
token.fit_on_texts(x_train)

total_words = len(token.word_index) + 1

x_train = token.texts_to_sequences(x_train)
x_test  = token.texts_to_sequences(x_test)

x_train = pad_sequences(x_train, maxlen=max_length, padding='post')
x_test  = pad_sequences(x_test, maxlen=max_length, padding='post')


In [18]:
from tensorflow.keras.layers import Dropout
EMBED_DIM = 64        # Increased embedding size
LSTM_OUT = 64

model = Sequential()

# Embedding Layer
model.add(Embedding(input_dim=total_words, 
                    output_dim=EMBED_DIM, 
                    input_length=max_length))

# First LSTM Layer (must return sequences for stacking)
model.add(LSTM(LSTM_OUT, return_sequences=True))
model.add(Dropout(0.3))   # Prevent overfitting

# Second LSTM Layer
model.add(LSTM(LSTM_OUT))
model.add(Dropout(0.3))

# Dense Layers
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))


E:\Anaconda installer\Lib\site-packages\keras\src\layers\core\embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [19]:
# #building lstm model
# EMBED_DIM = 32
# LSTM_OUT = 64

# model = Sequential()
# model.add(Embedding(total_words, EMBED_DIM, input_length=max_length))
# model.add(LSTM(LSTM_OUT))
# model.add(Dense(1, activation='sigmoid'))


In [20]:
#model compiler 
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [21]:
#model training
checkpoint = ModelCheckpoint(
    'models/LSTM.h5',
    monitor='accuracy',
    save_best_only=True,
    verbose=1
)

model.fit(
    x_train,
    y_train,
    batch_size=128,
    epochs=3,
    callbacks=[checkpoint]
)

Epoch 1/3
79/79 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.7038 - loss: 0.6069
Epoch 1: accuracy improved from None to 0.74427, saving model to models/LSTM.h5



Epoch 1: finished saving model to models/LSTM.h5
79/79 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.7443 - loss: 0.5391
Epoch 2/3
77/79 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8516 - loss: 0.3634
Epoch 2: accuracy improved from 0.74427 to 0.85207, saving model to models/LSTM.h5



Epoch 2: finished saving model to models/LSTM.h5
79/79 ━━━━━━━━━━━━━━━━━━━━ 2s 27ms/step - accuracy: 0.8521 - loss: 0.3508
Epoch 3/3
77/79 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.9030 - loss: 0.2500
Epoch 3: accuracy improved from 0.85207 to 0.89701, saving model to models/LSTM.h5



Epoch 3: finished saving model to models/LSTM.h5
79/79 ━━━━━━━━━━━━━━━━━━━━ 3s 27ms/step - accuracy: 0.8970 - loss: 0.2569


In [22]:
#model testing 
pred = model.predict(x_test)
y_pred = (pred >= 0.5).astype(int)
correct = 0
for i, y in enumerate(y_test):
    if y == y_pred[i]:
        correct += 1

print("Accuracy:", correct / len(y_test))


105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step
Accuracy: 0.8180726508555989
